In [1]:
import pandas as pd
import numpy as np
import os
import time

from sklearn.neighbors import BallTree

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

EARTH_RADIUS_KM = 6371.0088

VIIRS_FILE = "viirs_poc_detection_event_mapping.csv"

OSM_FILES = [
    r"C:\Users\vinja\Desktop\SIH\industrial-types.osm.pbf",
    r"C:\Users\vinja\Desktop\SIH\industrial-works.osm.pbf",
    r"C:\Users\vinja\Desktop\SIH\industrial-zones.osm.pbf"
]

RADIUS_1KM = 1.0
RADIUS_3KM = 3.0

print("V8 configuration loaded.")

V8 configuration loaded.


In [2]:
event_data = pd.read_csv(VIIRS_FILE)

event_data["acq_date"] = pd.to_datetime(event_data["acq_date"])

print("=" * 60)
print("VIIRS EVENT DATA")
print("=" * 60)
print("Detections:", len(event_data))
print("Candidate events:", event_data["event_id"].nunique())
print("Missing event IDs:", event_data["event_id"].isna().sum())

print("\nColumns:")
print(event_data.columns.tolist())

VIIRS EVENT DATA
Detections: 13609
Candidate events: 3337
Missing event IDs: 0

Columns:
['acq_date', 'daily_object_id', 'event_id', 'latitude', 'longitude', 'frp', 'bright_ti4', 'bright_ti5']


In [3]:
thermal_features = (
    event_data
    .groupby("event_id")
    .agg(
        mean_frp=("frp", "mean"),
        max_frp=("frp", "max"),
        std_frp=("frp", "std"),
        mean_bright_ti4=("bright_ti4", "mean"),
        max_bright_ti4=("bright_ti4", "max"),
        std_bright_ti4=("bright_ti4", "std"),
        mean_bright_ti5=("bright_ti5", "mean"),
        max_bright_ti5=("bright_ti5", "max"),
        std_bright_ti5=("bright_ti5", "std")
    )
    .reset_index()
    .fillna(0)
)

temporal_features = (
    event_data
    .groupby("event_id")
    .agg(
        start_date=("acq_date", "min"),
        end_date=("acq_date", "max"),
        active_days=("acq_date", "nunique"),
        detection_count=("event_id", "size")
    )
    .reset_index()
)

temporal_features["duration_days"] = (
    temporal_features["end_date"]
    - temporal_features["start_date"]
).dt.days + 1

temporal_features["activity_frequency"] = (
    temporal_features["active_days"]
    / temporal_features["duration_days"]
)

spatial_features = (
    event_data
    .groupby("event_id")
    .agg(
        centroid_lat=("latitude", "mean"),
        centroid_lon=("longitude", "mean")
    )
    .reset_index()
)

In [4]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))


event_points = event_data.merge(
    spatial_features,
    on="event_id",
    how="left",
    suffixes=("", "_event")
)

event_points["distance_from_centroid_km"] = haversine_km(
    event_points["latitude"],
    event_points["longitude"],
    event_points["centroid_lat"],
    event_points["centroid_lon"]
)

extent_features = (
    event_points
    .groupby("event_id")["distance_from_centroid_km"]
    .max()
    .reset_index(name="spatial_extent_km")
)

In [5]:
event_features = (
    thermal_features
    .merge(temporal_features, on="event_id", how="left")
    .merge(spatial_features, on="event_id", how="left")
    .merge(extent_features, on="event_id", how="left")
)

event_features["frp_range"] = (
    event_features["max_frp"]
    - event_features["mean_frp"]
)

event_features["ti4_range"] = (
    event_features["max_bright_ti4"]
    - event_features["mean_bright_ti4"]
)

event_features["ti5_range"] = (
    event_features["max_bright_ti5"]
    - event_features["mean_bright_ti5"]
)

event_features["detections_per_active_day"] = (
    event_features["detection_count"]
    / event_features["active_days"]
)

print("Event-level VIIRS table:")
print("Rows:", len(event_features))
print("Columns:", len(event_features.columns))

display(event_features.head())

Event-level VIIRS table:
Rows: 3337
Columns: 23


,event_id,mean_frp,max_frp,std_frp,mean_bright_ti4,max_bright_ti4,std_bright_ti4,mean_bright_ti5,max_bright_ti5,std_bright_ti5,start_date,end_date,active_days,detection_count,duration_days,activity_frequency,centroid_lat,centroid_lon,spatial_extent_km,frp_range,ti4_range,ti5_range,detections_per_active_day
0,1,2.0950,2.2100,0.1626,301.0900,304.9900,5.5154,277.2550,277.2700,0.0212,2024-01-01,2024-01-01,1,2,1,1.0000,30.5579,79.1196,0.0614,0.1150,3.9000,0.0150,2.0000
1,2,1.6600,1.6600,0.0000,309.7600,309.7600,0.0000,277.5900,277.5900,0.0000,2024-01-01,2024-01-01,1,1,1,1.0000,30.0443,80.5822,0.0000,0.0000,0.0000,0.0000,1.0000
2,3,0.9727,2.0200,0.3573,305.5497,316.6600,5.6908,282.8742,285.1700,1.9206,2024-01-01,2024-01-29,23,33,29,0.7931,27.4695,95.4220,0.3396,1.0473,11.1103,2.2958,1.4348
3,4,1.9026,4.1900,0.9092,311.6979,328.6800,9.1680,288.7890,292.2800,2.4260,2024-01-01,2024-01-17,17,39,17,1.0000,21.7580,83.8424,0.6851,2.2874,16.9821,3.4910,2.2941
4,5,1.2100,1.2100,0.0000,300.0300,300.0300,0.0000,289.4000,289.4000,0.0000,2024-01-01,2024-01-01,1,1,1,1.0000,21.7341,83.9805,0.0000,0.0000,0.0000,0.0000,1.0000


In [9]:
OSM_OUTPUT = "osm_points_v7.csv"

osm_points = pd.read_csv(OSM_OUTPUT)

print("=" * 60)
print("OSM POINT FEATURES")
print("=" * 60)
print("Rows:", len(osm_points))
print("Columns:", len(osm_points.columns))

display(osm_points.head())

OSM POINT FEATURES
Rows: 1981
Columns: 15


,osm_type,osm_id,latitude,longitude,industrial,landuse,power,man_made,building,product,plant_source,plant_method,resource,description,source_file
0,node,343703676,10.3377,76.2219,slaughterhouse,industrial,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,industrial-types.osm.pbf
1,node,1545193206,23.2396,69.7915,business,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,industrial-types.osm.pbf
2,node,2527030591,13.2387,80.0982,NaN,NaN,tower,NaN,NaN,NaN,NaN,NaN,NaN,NaN,industrial-types.osm.pbf
3,node,2556368813,10.7120,79.5162,sawmill,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,industrial-types.osm.pbf
4,node,3358197637,12.9706,74.8408,depot,industrial,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,industrial-types.osm.pbf


In [10]:
print("=" * 60)
print("OSM DATA VALIDATION")
print("=" * 60)

print("Missing latitude:", osm_points["latitude"].isna().sum())
print("Missing longitude:", osm_points["longitude"].isna().sum())

print(
    "Duplicate OSM IDs:",
    osm_points.duplicated(
        subset=["osm_type", "osm_id"]
    ).sum()
)

print("\nFeatures by source:")
display(
    osm_points["source_file"]
    .value_counts()
)

OSM DATA VALIDATION
Missing latitude: 0
Missing longitude: 0
Duplicate OSM IDs: 118

Features by source:


source_file
industrial-works.osm.pbf    1553
industrial-zones.osm.pbf     257
industrial-types.osm.pbf     171
Name: count, dtype: int64

In [11]:
before = len(osm_points)

osm_points = (
    osm_points
    .drop_duplicates(
        subset=["osm_type", "osm_id"]
    )
    .reset_index(drop=True)
)

after = len(osm_points)

print("Before deduplication:", before)
print("After deduplication:", after)
print("Duplicates removed:", before - after)

Before deduplication: 1981
After deduplication: 1863
Duplicates removed: 118


In [12]:
osm_points["is_industrial_zone"] = (
    osm_points["landuse"].eq("industrial")
).astype(int)

osm_points["is_factory"] = (
    osm_points["industrial"].eq("factory")
).astype(int)

osm_points["is_brickyard"] = (
    osm_points["industrial"].eq("brickyard")
).astype(int)

osm_points["is_brickworks"] = (
    osm_points["industrial"].eq("brickworks")
).astype(int)

osm_points["is_mine"] = (
    osm_points["industrial"].eq("mine")
).astype(int)

osm_points["is_depot"] = (
    osm_points["industrial"].isin(
        ["depot", "bus_depot"]
    )
).astype(int)

osm_points["is_cooling"] = (
    osm_points["industrial"].eq("cooling")
).astype(int)

osm_points["is_port"] = (
    osm_points["industrial"].eq("port")
).astype(int)

osm_points["is_power_plant"] = (
    osm_points["power"].eq("plant")
).astype(int)

osm_points["is_works"] = (
    osm_points["man_made"].eq("works")
).astype(int)

osm_points["is_kiln"] = (
    osm_points["man_made"].eq("kiln")
).astype(int)

osm_points["is_industrial_feature"] = (
    (
        osm_points["industrial"].notna()
    )
    |
    (
        osm_points["landuse"].eq("industrial")
    )
    |
    (
        osm_points["power"].eq("plant")
    )
    |
    (
        osm_points["man_made"].isin(["works", "kiln"])
    )
).astype(int)

osm_indicator_columns = [
    "is_industrial_zone",
    "is_factory",
    "is_brickyard",
    "is_brickworks",
    "is_mine",
    "is_depot",
    "is_cooling",
    "is_port",
    "is_power_plant",
    "is_works",
    "is_kiln",
    "is_industrial_feature"
]

display(
    osm_points[osm_indicator_columns].sum()
    .sort_values(ascending=False)
)

is_industrial_feature    1855
is_works                 1552
is_industrial_zone        249
is_depot                   20
is_factory                  8
is_brickyard                4
is_mine                     1
is_brickworks               0
is_port                     0
is_cooling                  0
is_power_plant              0
is_kiln                     0
dtype: int64

In [13]:
osm_summary = pd.DataFrame({
    "feature": osm_indicator_columns,
    "count": [
        osm_points[col].sum()
        for col in osm_indicator_columns
    ]
})

osm_summary["percentage_of_osm_points"] = (
    osm_summary["count"]
    / len(osm_points)
    * 100
)

display(
    osm_summary.sort_values(
        "count",
        ascending=False
    )
)

,feature,count,percentage_of_osm_points
11,is_industrial_feature,1855,99.5706
9,is_works,1552,83.3065
0,is_industrial_zone,249,13.3655
5,is_depot,20,1.0735
1,is_factory,8,0.4294
2,is_brickyard,4,0.2147
4,is_mine,1,0.0537
3,is_brickworks,0,0.0000
7,is_port,0,0.0000
6,is_cooling,0,0.0000


In [14]:
osm_coords_rad = np.radians(
    osm_points[
        ["latitude", "longitude"]
    ].to_numpy()
)

event_coords_rad = np.radians(
    event_features[
        ["centroid_lat", "centroid_lon"]
    ].to_numpy()
)

osm_tree = BallTree(
    osm_coords_rad,
    metric="haversine"
)

print("OSM spatial index created.")
print("OSM points:", len(osm_points))
print("VIIRS events:", len(event_features))

OSM spatial index created.
OSM points: 1863
VIIRS events: 3337


In [15]:
nearest_distance_rad, nearest_index = osm_tree.query(
    event_coords_rad,
    k=1
)

event_features["nearest_osm_distance_km"] = (
    nearest_distance_rad[:, 0]
    * EARTH_RADIUS_KM
)

print(
    event_features[
        "nearest_osm_distance_km"
    ].describe()
)

count   3337.0000
mean      53.0812
std       40.8027
min        0.1298
25%       24.7977
50%       48.2853
75%       74.1086
max     1167.4031
Name: nearest_osm_distance_km, dtype: float64


In [16]:
radius_1km_rad = RADIUS_1KM / EARTH_RADIUS_KM

neighbors_1km = osm_tree.query_radius(
    event_coords_rad,
    r=radius_1km_rad
)

print(
    "Events with at least one OSM feature within 1 km:",
    sum(len(x) > 0 for x in neighbors_1km)
)

print(
    "Events with no OSM feature within 1 km:",
    sum(len(x) == 0 for x in neighbors_1km)
)

Events with at least one OSM feature within 1 km: 10
Events with no OSM feature within 1 km: 3327


In [17]:
for column in osm_indicator_columns:
    
    feature_name = column.replace(
        "is_",
        ""
    ) + "_count_1km"
    
    event_features[feature_name] = [
        osm_points.iloc[idx][column].sum()
        if len(idx) > 0
        else 0
        for idx in neighbors_1km
    ]

print("1 km OSM features created.")

display(
    event_features[
        [
            "event_id",
            "industrial_feature_count_1km",
            "factory_count_1km",
            "brickyard_count_1km",
            "mine_count_1km",
            "power_plant_count_1km",
            "works_count_1km"
        ]
    ].head(10)
)

1 km OSM features created.


,event_id,industrial_feature_count_1km,factory_count_1km,brickyard_count_1km,mine_count_1km,power_plant_count_1km,works_count_1km
0,1,0,0,0,0,0,0
1,2,0,0,0,0,0,0
2,3,0,0,0,0,0,0
3,4,0,0,0,0,0,0
4,5,0,0,0,0,0,0
5,6,0,0,0,0,0,0
6,7,0,0,0,0,0,0
7,8,0,0,0,0,0,0
8,9,0,0,0,0,0,0
9,10,0,0,0,0,0,0


In [18]:
radius_3km_rad = RADIUS_3KM / EARTH_RADIUS_KM

neighbors_3km = osm_tree.query_radius(
    event_coords_rad,
    r=radius_3km_rad
)

print(
    "Events with at least one OSM feature within 3 km:",
    sum(len(x) > 0 for x in neighbors_3km)
)

print(
    "Events with no OSM feature within 3 km:",
    sum(len(x) == 0 for x in neighbors_3km)
)

Events with at least one OSM feature within 3 km: 64
Events with no OSM feature within 3 km: 3273


In [19]:
for column in osm_indicator_columns:
    
    feature_name = column.replace(
        "is_",
        ""
    ) + "_count_3km"
    
    event_features[feature_name] = [
        osm_points.iloc[idx][column].sum()
        if len(idx) > 0
        else 0
        for idx in neighbors_3km
    ]

print("3 km OSM features created.")

display(
    event_features[
        [
            "event_id",
            "industrial_feature_count_3km",
            "factory_count_3km",
            "brickyard_count_3km",
            "mine_count_3km",
            "power_plant_count_3km",
            "works_count_3km"
        ]
    ].head(10)
)

3 km OSM features created.


,event_id,industrial_feature_count_3km,factory_count_3km,brickyard_count_3km,mine_count_3km,power_plant_count_3km,works_count_3km
0,1,0,0,0,0,0,0
1,2,0,0,0,0,0,0
2,3,0,0,0,0,0,0
3,4,0,0,0,0,0,0
4,5,0,0,0,0,0,0
5,6,0,0,0,0,0,0
6,7,0,0,0,0,0,0
7,8,0,0,0,0,0,0
8,9,0,0,0,0,0,0
9,10,0,0,0,0,0,0


In [20]:
osm_context_columns = [
    col for col in event_features.columns
    if "_count_1km" in col
    or "_count_3km" in col
]

print("=" * 60)
print("OSM CONTEXT COVERAGE")
print("=" * 60)

coverage = pd.DataFrame({
    "feature": osm_context_columns,
    "events_with_feature": [
        (event_features[col] > 0).sum()
        for col in osm_context_columns
    ]
})

coverage["percentage"] = (
    coverage["events_with_feature"]
    / len(event_features)
    * 100
)

display(
    coverage.sort_values(
        "events_with_feature",
        ascending=False
    )
)

OSM CONTEXT COVERAGE


,feature,events_with_feature,percentage
23,industrial_feature_count_3km,64,1.9179
21,works_count_3km,64,1.9179
11,industrial_feature_count_1km,10,0.2997
9,works_count_1km,10,0.2997
12,industrial_zone_count_3km,3,0.0899
17,depot_count_3km,1,0.0300
0,industrial_zone_count_1km,0,0.0000
1,factory_count_1km,0,0.0000
7,port_count_1km,0,0.0000
6,cooling_count_1km,0,0.0000


In [21]:
print("=" * 60)
print("NEAREST OSM DISTANCE")
print("=" * 60)

display(
    event_features[
        "nearest_osm_distance_km"
    ].describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

NEAREST OSM DISTANCE


count   3337.0000
mean      53.0812
std       40.8027
min        0.1298
25%       24.7977
50%       48.2853
75%       74.1086
90%       98.0766
95%      113.0350
99%      178.8588
max     1167.4031
Name: nearest_osm_distance_km, dtype: float64

In [22]:
display(
    event_features[
        [
            "event_id",
            "centroid_lat",
            "centroid_lon",
            "duration_days",
            "active_days",
            "detection_count",
            "mean_frp",
            "max_frp",
            "spatial_extent_km",
            "nearest_osm_distance_km",
            "industrial_feature_count_1km",
            "factory_count_1km",
            "brickyard_count_1km",
            "mine_count_1km",
            "power_plant_count_1km",
            "works_count_1km",
            "industrial_feature_count_3km",
            "factory_count_3km",
            "brickyard_count_3km",
            "mine_count_3km",
            "power_plant_count_3km",
            "works_count_3km"
        ]
    ].head(20)
)

,event_id,centroid_lat,centroid_lon,duration_days,active_days,detection_count,mean_frp,max_frp,spatial_extent_km,nearest_osm_distance_km,industrial_feature_count_1km,factory_count_1km,brickyard_count_1km,mine_count_1km,power_plant_count_1km,works_count_1km,industrial_feature_count_3km,factory_count_3km,brickyard_count_3km,mine_count_3km,power_plant_count_3km,works_count_3km
0,1,30.5579,79.1196,1,1,2,2.0950,2.2100,0.0614,94.8509,0,0,0,0,0,0,0,0,0,0,0,0
1,2,30.0443,80.5822,1,1,1,1.6600,1.6600,0.0000,217.6613,0,0,0,0,0,0,0,0,0,0,0,0
2,3,27.4695,95.4220,29,23,33,0.9727,2.0200,0.3396,3.2230,0,0,0,0,0,0,0,0,0,0,0,0
3,4,21.7580,83.8424,17,17,39,1.9026,4.1900,0.6851,25.2635,0,0,0,0,0,0,0,0,0,0,0,0
4,5,21.7341,83.9805,1,1,1,1.2100,1.2100,0.0000,19.7006,0,0,0,0,0,0,0,0,0,0,0,0
5,6,21.6789,84.0389,17,17,44,2.0625,4.4100,0.7706,25.2232,0,0,0,0,0,0,0,0,0,0,0,0
6,7,21.4870,81.7687,31,29,46,1.5280,2.6700,0.3825,87.9381,0,0,0,0,0,0,0,0,0,0,0,0
7,8,20.9990,86.0126,2,2,3,1.0233,1.4000,0.2711,49.5079,0,0,0,0,0,0,0,0,0,0,0,0
8,9,20.9995,86.0263,7,6,7,0.6700,0.8900,0.3055,49.6509,0,0,0,0,0,0,0,0,0,0,0,0
9,10,21.0762,85.0394,15,15,94,1.8527,3.3700,1.3939,73.1421,0,0,0,0,0,0,0,0,0,0,0,0


In [23]:
missing_summary = (
    event_features
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_summary[
        missing_summary > 0
    ]
)

print(
    "\nTotal missing values:",
    event_features.isna().sum().sum()
)

Series([], dtype: int64)


Total missing values: 0


In [24]:
V8_OUTPUT = "viirs_poc_event_features_osm_v8.csv"

event_features.to_csv(
    V8_OUTPUT,
    index=False
)

print("=" * 60)
print("V8 DATASET SAVED")
print("=" * 60)
print("File:", V8_OUTPUT)
print("Rows:", len(event_features))
print("Columns:", len(event_features.columns))
print(
    "File size:",
    round(os.path.getsize(V8_OUTPUT) / 1024 / 1024, 2),
    "MB"
)

V8 DATASET SAVED
File: viirs_poc_event_features_osm_v8.csv
Rows: 3337
Columns: 48
File size: 0.78 MB


In [25]:
import pandas as pd
import numpy as np
import os
import time

from sklearn.neighbors import BallTree

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

EARTH_RADIUS_KM = 6371.0088

VIIRS_FILE = "viirs_poc_detection_event_mapping.csv"

OSM_FILE = "osm_points_v7.csv"

RADIUS_1KM = 1.0
RADIUS_3KM = 3.0

print("V8 configuration loaded.")

V8 configuration loaded.


In [26]:
event_data = pd.read_csv(VIIRS_FILE)

event_data["acq_date"] = pd.to_datetime(
    event_data["acq_date"]
)

print("=" * 60)
print("VIIRS EVENT DATA")
print("=" * 60)

print("Detections:", len(event_data))
print("Candidate events:", event_data["event_id"].nunique())
print("Missing event IDs:", event_data["event_id"].isna().sum())

print("\nColumns:")
print(event_data.columns.tolist())

VIIRS EVENT DATA
Detections: 13609
Candidate events: 3337
Missing event IDs: 0

Columns:
['acq_date', 'daily_object_id', 'event_id', 'latitude', 'longitude', 'frp', 'bright_ti4', 'bright_ti5']


In [27]:
thermal_features = (
    event_data
    .groupby("event_id")
    .agg(
        mean_frp=("frp", "mean"),
        max_frp=("frp", "max"),
        std_frp=("frp", "std"),

        mean_bright_ti4=("bright_ti4", "mean"),
        max_bright_ti4=("bright_ti4", "max"),
        std_bright_ti4=("bright_ti4", "std"),

        mean_bright_ti5=("bright_ti5", "mean"),
        max_bright_ti5=("bright_ti5", "max"),
        std_bright_ti5=("bright_ti5", "std")
    )
    .reset_index()
    .fillna(0)
)

print("=" * 60)
print("THERMAL EVENT FEATURES")
print("=" * 60)

print("Rows:", len(thermal_features))
print("Columns:", len(thermal_features.columns))

display(thermal_features.head())

THERMAL EVENT FEATURES
Rows: 3337
Columns: 10


,event_id,mean_frp,max_frp,std_frp,mean_bright_ti4,max_bright_ti4,std_bright_ti4,mean_bright_ti5,max_bright_ti5,std_bright_ti5
0,1,2.0950,2.2100,0.1626,301.0900,304.9900,5.5154,277.2550,277.2700,0.0212
1,2,1.6600,1.6600,0.0000,309.7600,309.7600,0.0000,277.5900,277.5900,0.0000
2,3,0.9727,2.0200,0.3573,305.5497,316.6600,5.6908,282.8742,285.1700,1.9206
3,4,1.9026,4.1900,0.9092,311.6979,328.6800,9.1680,288.7890,292.2800,2.4260
4,5,1.2100,1.2100,0.0000,300.0300,300.0300,0.0000,289.4000,289.4000,0.0000


In [28]:
temporal_features = (
    event_data
    .groupby("event_id")
    .agg(
        start_date=("acq_date", "min"),
        end_date=("acq_date", "max"),
        active_days=("acq_date", "nunique"),
        detection_count=("event_id", "size")
    )
    .reset_index()
)

temporal_features["duration_days"] = (
    temporal_features["end_date"]
    - temporal_features["start_date"]
).dt.days + 1

temporal_features["activity_frequency"] = (
    temporal_features["active_days"]
    / temporal_features["duration_days"]
)

print("=" * 60)
print("TEMPORAL EVENT FEATURES")
print("=" * 60)

print("Rows:", len(temporal_features))
print("Columns:", len(temporal_features.columns))

display(temporal_features.head())

TEMPORAL EVENT FEATURES
Rows: 3337
Columns: 7


,event_id,start_date,end_date,active_days,detection_count,duration_days,activity_frequency
0,1,2024-01-01,2024-01-01,1,2,1,1.0000
1,2,2024-01-01,2024-01-01,1,1,1,1.0000
2,3,2024-01-01,2024-01-29,23,33,29,0.7931
3,4,2024-01-01,2024-01-17,17,39,17,1.0000
4,5,2024-01-01,2024-01-01,1,1,1,1.0000


In [29]:
spatial_features = (
    event_data
    .groupby("event_id")
    .agg(
        centroid_lat=("latitude", "mean"),
        centroid_lon=("longitude", "mean")
    )
    .reset_index()
)

print("=" * 60)
print("SPATIAL EVENT FEATURES")
print("=" * 60)

print("Rows:", len(spatial_features))
print("Columns:", len(spatial_features.columns))

display(spatial_features.head())

SPATIAL EVENT FEATURES
Rows: 3337
Columns: 3


,event_id,centroid_lat,centroid_lon
0,1,30.5579,79.1196
1,2,30.0443,80.5822
2,3,27.4695,95.4220
3,4,21.7580,83.8424
4,5,21.7341,83.9805


In [30]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return 2 * EARTH_RADIUS_KM * np.arcsin(
        np.sqrt(a)
    )


event_points = event_data.merge(
    spatial_features,
    on="event_id",
    how="left"
)

event_points["distance_from_centroid_km"] = haversine_km(
    event_points["latitude"],
    event_points["longitude"],
    event_points["centroid_lat"],
    event_points["centroid_lon"]
)

extent_features = (
    event_points
    .groupby("event_id")["distance_from_centroid_km"]
    .max()
    .reset_index(name="spatial_extent_km")
)

print("Spatial extent calculated.")
print(
    extent_features["spatial_extent_km"].describe()
)

Spatial extent calculated.
count   3337.0000
mean       0.1497
std        0.2778
min        0.0000
25%        0.0000
50%        0.0000
75%        0.2135
max        2.9703
Name: spatial_extent_km, dtype: float64


In [31]:
event_features = (
    thermal_features
    .merge(
        temporal_features,
        on="event_id",
        how="left"
    )
    .merge(
        spatial_features,
        on="event_id",
        how="left"
    )
    .merge(
        extent_features,
        on="event_id",
        how="left"
    )
)

event_features["frp_range"] = (
    event_features["max_frp"]
    - event_features["mean_frp"]
)

event_features["ti4_range"] = (
    event_features["max_bright_ti4"]
    - event_features["mean_bright_ti4"]
)

event_features["ti5_range"] = (
    event_features["max_bright_ti5"]
    - event_features["mean_bright_ti5"]
)

event_features["detections_per_active_day"] = (
    event_features["detection_count"]
    / event_features["active_days"]
)

print("=" * 60)
print("VIIRS EVENT FEATURE TABLE")
print("=" * 60)

print("Rows:", len(event_features))
print("Columns:", len(event_features.columns))

display(event_features.head())

VIIRS EVENT FEATURE TABLE
Rows: 3337
Columns: 23


,event_id,mean_frp,max_frp,std_frp,mean_bright_ti4,max_bright_ti4,std_bright_ti4,mean_bright_ti5,max_bright_ti5,std_bright_ti5,start_date,end_date,active_days,detection_count,duration_days,activity_frequency,centroid_lat,centroid_lon,spatial_extent_km,frp_range,ti4_range,ti5_range,detections_per_active_day
0,1,2.0950,2.2100,0.1626,301.0900,304.9900,5.5154,277.2550,277.2700,0.0212,2024-01-01,2024-01-01,1,2,1,1.0000,30.5579,79.1196,0.0614,0.1150,3.9000,0.0150,2.0000
1,2,1.6600,1.6600,0.0000,309.7600,309.7600,0.0000,277.5900,277.5900,0.0000,2024-01-01,2024-01-01,1,1,1,1.0000,30.0443,80.5822,0.0000,0.0000,0.0000,0.0000,1.0000
2,3,0.9727,2.0200,0.3573,305.5497,316.6600,5.6908,282.8742,285.1700,1.9206,2024-01-01,2024-01-29,23,33,29,0.7931,27.4695,95.4220,0.3396,1.0473,11.1103,2.2958,1.4348
3,4,1.9026,4.1900,0.9092,311.6979,328.6800,9.1680,288.7890,292.2800,2.4260,2024-01-01,2024-01-17,17,39,17,1.0000,21.7580,83.8424,0.6851,2.2874,16.9821,3.4910,2.2941
4,5,1.2100,1.2100,0.0000,300.0300,300.0300,0.0000,289.4000,289.4000,0.0000,2024-01-01,2024-01-01,1,1,1,1.0000,21.7341,83.9805,0.0000,0.0000,0.0000,0.0000,1.0000


In [32]:
osm_points = pd.read_csv(OSM_FILE)

print("=" * 60)
print("OSM POINT FEATURES")
print("=" * 60)

print("Rows:", len(osm_points))
print("Columns:", len(osm_points.columns))

display(osm_points.head())

OSM POINT FEATURES
Rows: 1981
Columns: 15


,osm_type,osm_id,latitude,longitude,industrial,landuse,power,man_made,building,product,plant_source,plant_method,resource,description,source_file
0,node,343703676,10.3377,76.2219,slaughterhouse,industrial,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,industrial-types.osm.pbf
1,node,1545193206,23.2396,69.7915,business,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,industrial-types.osm.pbf
2,node,2527030591,13.2387,80.0982,NaN,NaN,tower,NaN,NaN,NaN,NaN,NaN,NaN,NaN,industrial-types.osm.pbf
3,node,2556368813,10.7120,79.5162,sawmill,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,industrial-types.osm.pbf
4,node,3358197637,12.9706,74.8408,depot,industrial,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,industrial-types.osm.pbf


In [33]:
print("=" * 60)
print("OSM VALIDATION")
print("=" * 60)

print(
    "Missing latitude:",
    osm_points["latitude"].isna().sum()
)

print(
    "Missing longitude:",
    osm_points["longitude"].isna().sum()
)

print(
    "Duplicate OSM objects:",
    osm_points.duplicated(
        subset=["osm_type", "osm_id"]
    ).sum()
)

print("\nFeatures by source:")
display(
    osm_points["source_file"]
    .value_counts()
)

OSM VALIDATION
Missing latitude: 0
Missing longitude: 0
Duplicate OSM objects: 118

Features by source:


source_file
industrial-works.osm.pbf    1553
industrial-zones.osm.pbf     257
industrial-types.osm.pbf     171
Name: count, dtype: int64

In [34]:
before = len(osm_points)

osm_points = (
    osm_points
    .drop_duplicates(
        subset=["osm_type", "osm_id"]
    )
    .reset_index(drop=True)
)

after = len(osm_points)

print("=" * 60)
print("OSM DEDUPLICATION")
print("=" * 60)

print("Before:", before)
print("After:", after)
print("Removed:", before - after)

OSM DEDUPLICATION
Before: 1981
After: 1863
Removed: 118


In [35]:
osm_tag_columns = [
    "industrial",
    "landuse",
    "power",
    "man_made",
    "building",
    "product"
]

for column in osm_tag_columns:

    print("\n" + "=" * 60)
    print(column.upper())
    print("=" * 60)

    display(
        osm_points[column]
        .value_counts(dropna=True)
        .head(20)
    )


INDUSTRIAL


industrial
slaughterhouse            58
scrap_yard                24
depot                     20
Pharmaceutical Company    14
factory                    8
grinding_mill              7
warehouse                  5
brickyard                  4
Biotechnology Company      3
concrete_plant             3
Pharmaceutical company     3
oil                        2
rice_mill                  2
Research Company           2
sawmill                    2
oil_mill                   2
mine                       1
business                   1
mill                       1
food_industry              1
Name: count, dtype: int64


LANDUSE


landuse
industrial    249
commercial      2
quarry          1
Name: count, dtype: int64


POWER


...

man_made
works               1552
wastewater_plant       1
Name: count, dtype: int64


BUILDING


building
industrial     7
yes            4
office         1
residential    1
Name: count, dtype: int64


PRODUCT


product
beer              27
oxygen            25
food              20
rice              14
bricks            11
furniture         10
machinery          6
plastic            5
oil                4
clothes            4
flour              4
tea                3
dairy              3
concrete           3
cotton             2
electronics        2
food;ice_cream     2
sugar              2
metal              2
cashew             2
Name: count, dtype: int64

In [36]:
osm_points["is_industrial_zone"] = (
    osm_points["landuse"].eq("industrial")
).astype(int)

osm_points["is_factory"] = (
    osm_points["industrial"].eq("factory")
).astype(int)

osm_points["is_brickyard"] = (
    osm_points["industrial"].eq("brickyard")
).astype(int)

osm_points["is_brickworks"] = (
    osm_points["industrial"].eq("brickworks")
).astype(int)

osm_points["is_mine"] = (
    osm_points["industrial"].eq("mine")
).astype(int)

osm_points["is_depot"] = (
    osm_points["industrial"].isin(
        ["depot", "bus_depot"]
    )
).astype(int)

osm_points["is_cooling"] = (
    osm_points["industrial"].eq("cooling")
).astype(int)

osm_points["is_port"] = (
    osm_points["industrial"].eq("port")
).astype(int)

osm_points["is_power_plant"] = (
    osm_points["power"].eq("plant")
).astype(int)

osm_points["is_works"] = (
    osm_points["man_made"].eq("works")
).astype(int)

osm_points["is_kiln"] = (
    osm_points["man_made"].eq("kiln")
).astype(int)

In [37]:
osm_indicator_columns = [
    "is_industrial_zone",
    "is_factory",
    "is_brickyard",
    "is_brickworks",
    "is_mine",
    "is_depot",
    "is_cooling",
    "is_port",
    "is_power_plant",
    "is_works",
    "is_kiln"
]

osm_summary = pd.DataFrame({
    "feature": osm_indicator_columns,
    "count": [
        osm_points[column].sum()
        for column in osm_indicator_columns
    ]
})

osm_summary["percentage"] = (
    osm_summary["count"]
    / len(osm_points)
    * 100
)

display(
    osm_summary.sort_values(
        "count",
        ascending=False
    )
)

,feature,count,percentage
9,is_works,1552,83.3065
0,is_industrial_zone,249,13.3655
5,is_depot,20,1.0735
1,is_factory,8,0.4294
2,is_brickyard,4,0.2147
4,is_mine,1,0.0537
3,is_brickworks,0,0.0000
6,is_cooling,0,0.0000
7,is_port,0,0.0000
8,is_power_plant,0,0.0000


In [38]:
osm_points["is_industrial_feature"] = (
    osm_points[osm_indicator_columns]
    .max(axis=1)
)

print(
    "Industrial/context OSM features:",
    osm_points["is_industrial_feature"].sum()
)

print(
    "Total OSM features:",
    len(osm_points)
)

Industrial/context OSM features: 1809
Total OSM features: 1863


In [39]:
osm_coords_rad = np.radians(
    osm_points[
        ["latitude", "longitude"]
    ].to_numpy()
)

event_coords_rad = np.radians(
    event_features[
        ["centroid_lat", "centroid_lon"]
    ].to_numpy()
)

print("OSM coordinate array:", osm_coords_rad.shape)
print("VIIRS coordinate array:", event_coords_rad.shape)

OSM coordinate array: (1863, 2)
VIIRS coordinate array: (3337, 2)


In [40]:
osm_tree = BallTree(
    osm_coords_rad,
    metric="haversine"
)

print("=" * 60)
print("BALLTREE CREATED")
print("=" * 60)

print("OSM spatial points:", len(osm_points))
print("VIIRS events:", len(event_features))

BALLTREE CREATED
OSM spatial points: 1863
VIIRS events: 3337


In [41]:
nearest_distance_rad, nearest_index = osm_tree.query(
    event_coords_rad,
    k=1
)

event_features["nearest_osm_distance_km"] = (
    nearest_distance_rad[:, 0]
    * EARTH_RADIUS_KM
)

print("=" * 60)
print("NEAREST OSM DISTANCE")
print("=" * 60)

display(
    event_features[
        "nearest_osm_distance_km"
    ].describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

NEAREST OSM DISTANCE


count   3337.0000
mean      53.0812
std       40.8027
min        0.1298
25%       24.7977
50%       48.2853
75%       74.1086
90%       98.0766
95%      113.0350
99%      178.8588
max     1167.4031
Name: nearest_osm_distance_km, dtype: float64

In [42]:
radius_1km_rad = (
    RADIUS_1KM
    / EARTH_RADIUS_KM
)

neighbors_1km = osm_tree.query_radius(
    event_coords_rad,
    r=radius_1km_rad
)

events_with_osm_1km = sum(
    len(indices) > 0
    for indices in neighbors_1km
)

print("=" * 60)
print("1 KM OSM CONTEXT")
print("=" * 60)

print(
    "Events with OSM context:",
    events_with_osm_1km
)

print(
    "Events without OSM context:",
    len(event_features)
    - events_with_osm_1km
)

1 KM OSM CONTEXT
Events with OSM context: 10
Events without OSM context: 3327


In [43]:
for column in osm_indicator_columns:

    feature_name = (
        column.replace("is_", "")
        + "_count_1km"
    )

    event_features[feature_name] = [
        osm_points.iloc[indices][column].sum()
        if len(indices) > 0
        else 0
        for indices in neighbors_1km
    ]

# Overall industrial/context count
event_features["industrial_feature_count_1km"] = [
    osm_points.iloc[indices][
        "is_industrial_feature"
    ].sum()
    if len(indices) > 0
    else 0
    for indices in neighbors_1km
]

print("1 km contextual features created.")

display(
    event_features[
        [
            "event_id",
            "industrial_feature_count_1km",
            "factory_count_1km",
            "brickyard_count_1km",
            "mine_count_1km",
            "power_plant_count_1km",
            "works_count_1km",
            "kiln_count_1km"
        ]
    ].head(10)
)

1 km contextual features created.


,event_id,industrial_feature_count_1km,factory_count_1km,brickyard_count_1km,mine_count_1km,power_plant_count_1km,works_count_1km,kiln_count_1km
0,1,0,0,0,0,0,0,0
1,2,0,0,0,0,0,0,0
2,3,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0
4,5,0,0,0,0,0,0,0
5,6,0,0,0,0,0,0,0
6,7,0,0,0,0,0,0,0
7,8,0,0,0,0,0,0,0
8,9,0,0,0,0,0,0,0
9,10,0,0,0,0,0,0,0


In [44]:
radius_3km_rad = (
    RADIUS_3KM
    / EARTH_RADIUS_KM
)

neighbors_3km = osm_tree.query_radius(
    event_coords_rad,
    r=radius_3km_rad
)

events_with_osm_3km = sum(
    len(indices) > 0
    for indices in neighbors_3km
)

print("=" * 60)
print("3 KM OSM CONTEXT")
print("=" * 60)

print(
    "Events with OSM context:",
    events_with_osm_3km
)

print(
    "Events without OSM context:",
    len(event_features)
    - events_with_osm_3km
)

3 KM OSM CONTEXT
Events with OSM context: 64
Events without OSM context: 3273


In [45]:
for column in osm_indicator_columns:

    feature_name = (
        column.replace("is_", "")
        + "_count_3km"
    )

    event_features[feature_name] = [
        osm_points.iloc[indices][column].sum()
        if len(indices) > 0
        else 0
        for indices in neighbors_3km
    ]

event_features["industrial_feature_count_3km"] = [
    osm_points.iloc[indices][
        "is_industrial_feature"
    ].sum()
    if len(indices) > 0
    else 0
    for indices in neighbors_3km
]

print("3 km contextual features created.")

display(
    event_features[
        [
            "event_id",
            "industrial_feature_count_3km",
            "factory_count_3km",
            "brickyard_count_3km",
            "mine_count_3km",
            "power_plant_count_3km",
            "works_count_3km",
            "kiln_count_3km"
        ]
    ].head(10)
)

3 km contextual features created.


,event_id,industrial_feature_count_3km,factory_count_3km,brickyard_count_3km,mine_count_3km,power_plant_count_3km,works_count_3km,kiln_count_3km
0,1,0,0,0,0,0,0,0
1,2,0,0,0,0,0,0,0
2,3,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0
4,5,0,0,0,0,0,0,0
5,6,0,0,0,0,0,0,0
6,7,0,0,0,0,0,0,0
7,8,0,0,0,0,0,0,0
8,9,0,0,0,0,0,0,0
9,10,0,0,0,0,0,0,0


In [46]:
osm_context_columns = [
    column
    for column in event_features.columns
    if "_count_1km" in column
    or "_count_3km" in column
]

coverage = pd.DataFrame({
    "feature": osm_context_columns,
    "events_with_feature": [
        (
            event_features[column] > 0
        ).sum()
        for column in osm_context_columns
    ]
})

coverage["percentage"] = (
    coverage["events_with_feature"]
    / len(event_features)
    * 100
)

print("=" * 60)
print("OSM CONTEXT COVERAGE")
print("=" * 60)

display(
    coverage.sort_values(
        "events_with_feature",
        ascending=False
    )
)

OSM CONTEXT COVERAGE


,feature,events_with_feature,percentage
23,industrial_feature_count_3km,64,1.9179
21,works_count_3km,64,1.9179
11,industrial_feature_count_1km,10,0.2997
9,works_count_1km,10,0.2997
12,industrial_zone_count_3km,3,0.0899
17,depot_count_3km,1,0.0300
0,industrial_zone_count_1km,0,0.0000
1,factory_count_1km,0,0.0000
7,port_count_1km,0,0.0000
6,cooling_count_1km,0,0.0000


In [47]:
osm_count_summary = pd.DataFrame({
    "feature": osm_context_columns,
    "mean": [
        event_features[column].mean()
        for column in osm_context_columns
    ],
    "median": [
        event_features[column].median()
        for column in osm_context_columns
    ],
    "max": [
        event_features[column].max()
        for column in osm_context_columns
    ],
    "events_nonzero": [
        (
            event_features[column] > 0
        ).sum()
        for column in osm_context_columns
    ]
})

display(
    osm_count_summary.sort_values(
        "events_nonzero",
        ascending=False
    )
)

,feature,mean,median,max,events_nonzero
23,industrial_feature_count_3km,0.0351,0.0000,7,64
21,works_count_3km,0.0342,0.0000,6,64
11,industrial_feature_count_1km,0.0030,0.0000,1,10
9,works_count_1km,0.0030,0.0000,1,10
12,industrial_zone_count_3km,0.0009,0.0000,1,3
17,depot_count_3km,0.0003,0.0000,1,1
0,industrial_zone_count_1km,0.0000,0.0000,0,0
1,factory_count_1km,0.0000,0.0000,0,0
7,port_count_1km,0.0000,0.0000,0,0
6,cooling_count_1km,0.0000,0.0000,0,0


In [48]:
comparison = pd.DataFrame({
    "metric": [
        "Mean industrial features",
        "Median industrial features",
        "Maximum industrial features",
        "Events with industrial context"
    ],

    "1km": [
        event_features[
            "industrial_feature_count_1km"
        ].mean(),

        event_features[
            "industrial_feature_count_1km"
        ].median(),

        event_features[
            "industrial_feature_count_1km"
        ].max(),

        (
            event_features[
                "industrial_feature_count_1km"
            ] > 0
        ).sum()
    ],

    "3km": [
        event_features[
            "industrial_feature_count_3km"
        ].mean(),

        event_features[
            "industrial_feature_count_3km"
        ].median(),

        event_features[
            "industrial_feature_count_3km"
        ].max(),

        (
            event_features[
                "industrial_feature_count_3km"
            ] > 0
        ).sum()
    ]
})

display(comparison)

,metric,1km,3km
0,Mean industrial features,0.0030,0.0351
1,Median industrial features,0.0000,0.0000
2,Maximum industrial features,1.0000,7.0000
3,Events with industrial context,10.0000,64.0000


In [49]:
inspection_columns = [
    "event_id",
    "centroid_lat",
    "centroid_lon",

    "duration_days",
    "active_days",
    "detection_count",

    "mean_frp",
    "max_frp",
    "spatial_extent_km",

    "nearest_osm_distance_km",

    "industrial_feature_count_1km",
    "factory_count_1km",
    "brickyard_count_1km",
    "brickworks_count_1km",
    "mine_count_1km",
    "depot_count_1km",
    "cooling_count_1km",
    "port_count_1km",
    "power_plant_count_1km",
    "works_count_1km",
    "kiln_count_1km",

    "industrial_feature_count_3km",
    "factory_count_3km",
    "brickyard_count_3km",
    "brickworks_count_3km",
    "mine_count_3km",
    "depot_count_3km",
    "cooling_count_3km",
    "port_count_3km",
    "power_plant_count_3km",
    "works_count_3km",
    "kiln_count_3km"
]

display(
    event_features[
        inspection_columns
    ].head(20)
)

,event_id,centroid_lat,centroid_lon,duration_days,active_days,detection_count,mean_frp,max_frp,spatial_extent_km,nearest_osm_distance_km,industrial_feature_count_1km,factory_count_1km,brickyard_count_1km,brickworks_count_1km,mine_count_1km,depot_count_1km,cooling_count_1km,port_count_1km,power_plant_count_1km,works_count_1km,kiln_count_1km,industrial_feature_count_3km,factory_count_3km,brickyard_count_3km,brickworks_count_3km,mine_count_3km,depot_count_3km,cooling_count_3km,port_count_3km,power_plant_count_3km,works_count_3km,kiln_count_3km
0,1,30.5579,79.1196,1,1,2,2.0950,2.2100,0.0614,94.8509,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,30.0443,80.5822,1,1,1,1.6600,1.6600,0.0000,217.6613,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,3,27.4695,95.4220,29,23,33,0.9727,2.0200,0.3396,3.2230,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,4,21.7580,83.8424,17,17,39,1.9026,4.1900,0.6851,25.2635,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,5,21.7341,83.9805,1,1,1,1.2100,1.2100,0.0000,19.7006,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,6,21.6789,84.0389,17,17,44,2.0625,4.4100,0.7706,25.2232,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,7,21.4870,81.7687,31,29,46,1.5280,2.6700,0.3825,87.9381,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,8,20.9990,86.0126,2,2,3,1.0233,1.4000,0.2711,49.5079,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,9,20.9995,86.0263,7,6,7,0.6700,0.8900,0.3055,49.6509,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,10,21.0762,85.0394,15,15,94,1.8527,3.3700,1.3939,73.1421,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [50]:
missing_summary = (
    event_features
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

print("=" * 60)
print("MISSING VALUES")
print("=" * 60)

display(
    missing_summary[
        missing_summary > 0
    ]
)

print(
    "\nTotal missing values:",
    event_features.isna().sum().sum()
)

MISSING VALUES


Series([], dtype: int64)


Total missing values: 0


In [51]:
print("=" * 60)
print("V8 FEATURE SANITY CHECK")
print("=" * 60)

print("Events:", len(event_features))

print(
    "Unique event IDs:",
    event_features["event_id"].nunique()
)

print(
    "Minimum latitude:",
    event_features["centroid_lat"].min()
)

print(
    "Maximum latitude:",
    event_features["centroid_lat"].max()
)

print(
    "Minimum longitude:",
    event_features["centroid_lon"].min()
)

print(
    "Maximum longitude:",
    event_features["centroid_lon"].max()
)

print(
    "Nearest OSM distance range:",
    round(
        event_features[
            "nearest_osm_distance_km"
        ].min(),
        4
    ),
    "to",
    round(
        event_features[
            "nearest_osm_distance_km"
        ].max(),
        4
    ),
    "km"
)

V8 FEATURE SANITY CHECK
Events: 3337
Unique event IDs: 3337
Minimum latitude: 8.24339
Maximum latitude: 34.62992
Minimum longitude: 68.57577
Maximum longitude: 97.07545999999999
Nearest OSM distance range: 0.1298 to 1167.4031 km


In [52]:
V8_OUTPUT = (
    "viirs_poc_event_features_osm_v8.csv"
)

event_features.to_csv(
    V8_OUTPUT,
    index=False
)

print("=" * 60)
print("V8 DATASET SAVED")
print("=" * 60)

print("File:", V8_OUTPUT)
print("Rows:", len(event_features))
print("Columns:", len(event_features.columns))

print(
    "File size:",
    round(
        os.path.getsize(V8_OUTPUT)
        / 1024 / 1024,
        2
    ),
    "MB"
)

V8 DATASET SAVED
File: viirs_poc_event_features_osm_v8.csv
Rows: 3337
Columns: 48
File size: 0.78 MB


In [53]:
print("=" * 60)
print("FINAL V8 FEATURE INVENTORY")
print("=" * 60)

for i, column in enumerate(
    event_features.columns,
    start=1
):
    print(f"{i:02d}. {column}")

FINAL V8 FEATURE INVENTORY
01. event_id
02. mean_frp
03. max_frp
04. std_frp
05. mean_bright_ti4
06. max_bright_ti4
07. std_bright_ti4
08. mean_bright_ti5
09. max_bright_ti5
10. std_bright_ti5
11. start_date
12. end_date
13. active_days
14. detection_count
15. duration_days
16. activity_frequency
17. centroid_lat
18. centroid_lon
19. spatial_extent_km
20. frp_range
21. ti4_range
22. ti5_range
23. detections_per_active_day
24. nearest_osm_distance_km
25. industrial_zone_count_1km
26. factory_count_1km
27. brickyard_count_1km
28. brickworks_count_1km
29. mine_count_1km
30. depot_count_1km
31. cooling_count_1km
32. port_count_1km
33. power_plant_count_1km
34. works_count_1km
35. kiln_count_1km
36. industrial_feature_count_1km
37. industrial_zone_count_3km
38. factory_count_3km
39. brickyard_count_3km
40. brickworks_count_3km
41. mine_count_3km
42. depot_count_3km
43. cooling_count_3km
44. port_count_3km
45. power_plant_count_3km
46. works_count_3km
47. kiln_count_3km
48. industrial_featur

In [54]:
# ============================================================
# V8 — COMPLETE DIAGNOSTIC SUMMARY
# Run this as ONE cell
# ============================================================

import pandas as pd
import numpy as np

FILE = "viirs_poc_event_features_osm_v8.csv"

df = pd.read_csv(FILE)

print("=" * 70)
print("V8 COMPLETE DIAGNOSTIC SUMMARY")
print("=" * 70)

# ------------------------------------------------------------
# 1. DATASET OVERVIEW
# ------------------------------------------------------------
print("\n[1] DATASET OVERVIEW")
print("-" * 70)

print(f"Rows / events              : {len(df):,}")
print(f"Columns                    : {len(df.columns)}")
print(f"Missing values (total)     : {df.isna().sum().sum():,}")
print(f"Duplicate event IDs        : {df['event_id'].duplicated().sum():,}")

print("\nDate range:")
print(f"  Start                     : {df['start_date'].min()}")
print(f"  End                       : {df['end_date'].max()}")

# ------------------------------------------------------------
# 2. EVENT TEMPORAL CHARACTERISTICS
# ------------------------------------------------------------
print("\n[2] TEMPORAL EVENT CHARACTERISTICS")
print("-" * 70)

for col in [
    "active_days",
    "duration_days",
    "detection_count",
    "activity_frequency",
    "detections_per_active_day"
]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].describe().round(3).to_string())

print("\nPersistence thresholds:")
for threshold in [1, 2, 3, 5, 7, 10, 15, 20, 30]:
    count = (df["active_days"] >= threshold).sum()
    pct = count / len(df) * 100
    print(f"  >= {threshold:2d} active days : {count:5,} ({pct:6.2f}%)")

# ------------------------------------------------------------
# 3. THERMAL CHARACTERISTICS
# ------------------------------------------------------------
print("\n[3] THERMAL CHARACTERISTICS")
print("-" * 70)

thermal_cols = [
    "mean_frp",
    "max_frp",
    "std_frp",
    "mean_bright_ti4",
    "max_bright_ti4",
    "std_bright_ti4",
    "mean_bright_ti5",
    "max_bright_ti5",
    "std_bright_ti5",
    "frp_range",
    "ti4_range",
    "ti5_range"
]

existing_thermal = [c for c in thermal_cols if c in df.columns]

print(
    df[existing_thermal]
    .describe()
    .T
    .round(3)
    .to_string()
)

# ------------------------------------------------------------
# 4. SPATIAL CHARACTERISTICS
# ------------------------------------------------------------
print("\n[4] SPATIAL CHARACTERISTICS")
print("-" * 70)

spatial_cols = [
    "spatial_extent_km",
    "centroid_lat",
    "centroid_lon"
]

for col in spatial_cols:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].describe().round(4).to_string())

# ------------------------------------------------------------
# 5. OSM COVERAGE
# ------------------------------------------------------------
print("\n[5] OSM COVERAGE")
print("-" * 70)

osm_cols = [
    "nearest_osm_distance_km",
    "industrial_feature_count_1km",
    "industrial_feature_count_3km",
    "industrial_zone_count_1km",
    "industrial_zone_count_3km",
    "factory_count_1km",
    "factory_count_3km",
    "brickyard_count_1km",
    "brickyard_count_3km",
    "brickworks_count_1km",
    "brickworks_count_3km",
    "mine_count_1km",
    "mine_count_3km",
    "depot_count_1km",
    "depot_count_3km",
    "cooling_count_1km",
    "cooling_count_3km",
    "port_count_1km",
    "port_count_3km",
    "power_plant_count_1km",
    "power_plant_count_3km",
    "works_count_1km",
    "works_count_3km",
    "kiln_count_1km",
    "kiln_count_3km"
]

existing_osm = [c for c in osm_cols if c in df.columns]

print(
    df[existing_osm]
    .describe()
    .T
    .round(3)
    .to_string()
)

# ------------------------------------------------------------
# 6. OSM PRESENCE / ABSENCE
# ------------------------------------------------------------
print("\n[6] OSM PRESENCE")
print("-" * 70)

for radius in ["1km", "3km"]:

    col = f"industrial_feature_count_{radius}"

    if col not in df.columns:
        continue

    present = (df[col] > 0).sum()
    absent = (df[col] == 0).sum()

    print(
        f"{radius:>3} radius | "
        f"with industrial OSM = {present:5,} ({present/len(df)*100:6.2f}%) | "
        f"without = {absent:5,} ({absent/len(df)*100:6.2f}%)"
    )

# ------------------------------------------------------------
# 7. OSM COUNT DISTRIBUTIONS
# ------------------------------------------------------------
print("\n[7] OSM INDUSTRIAL FEATURE COUNT DISTRIBUTIONS")
print("-" * 70)

for radius in ["1km", "3km"]:

    col = f"industrial_feature_count_{radius}"

    if col not in df.columns:
        continue

    print(f"\n{radius}:")

    print(
        df[col]
        .value_counts()
        .sort_index()
        .head(20)
        .to_string()
    )

# ------------------------------------------------------------
# 8. 1 KM VS 3 KM COMPARISON
# ------------------------------------------------------------
print("\n[8] 1 KM VS 3 KM COMPARISON")
print("-" * 70)

comparison_base = [
    "industrial_feature_count",
    "industrial_zone_count",
    "factory_count",
    "brickyard_count",
    "brickworks_count",
    "mine_count",
    "depot_count",
    "cooling_count",
    "port_count",
    "power_plant_count",
    "works_count",
    "kiln_count"
]

comparison_rows = []

for base in comparison_base:

    c1 = f"{base}_1km"
    c3 = f"{base}_3km"

    if c1 in df.columns and c3 in df.columns:

        comparison_rows.append({
            "feature": base,
            "events_1km": int((df[c1] > 0).sum()),
            "events_3km": int((df[c3] > 0).sum()),
            "mean_1km": df[c1].mean(),
            "mean_3km": df[c3].mean(),
            "max_1km": df[c1].max(),
            "max_3km": df[c3].max()
        })

comparison = pd.DataFrame(comparison_rows)

if len(comparison):
    print(comparison.round(3).to_string(index=False))

# ------------------------------------------------------------
# 9. OSM FEATURES FOR PERSISTENT EVENTS
# ------------------------------------------------------------
print("\n[9] OSM CONTEXT VS EVENT PERSISTENCE")
print("-" * 70)

for threshold in [1, 3, 5, 7]:

    subset = df[df["active_days"] >= threshold]

    if len(subset) == 0:
        continue

    print(f"\nEvents with >= {threshold} active days: {len(subset):,}")

    for col in [
        "industrial_feature_count_1km",
        "industrial_feature_count_3km",
        "industrial_zone_count_1km",
        "factory_count_1km",
        "power_plant_count_1km",
        "works_count_1km",
        "kiln_count_1km",
        "nearest_osm_distance_km"
    ]:

        if col not in subset.columns:
            continue

        print(
            f"  {col:<35} "
            f"mean={subset[col].mean():8.3f} | "
            f"median={subset[col].median():8.3f}"
        )

# ------------------------------------------------------------
# 10. HIGH-PERSISTENCE + OSM CASES
# ------------------------------------------------------------
print("\n[10] EXAMPLE HIGH-PERSISTENCE EVENTS")
print("-" * 70)

display_cols = [
    "event_id",
    "active_days",
    "duration_days",
    "detection_count",
    "mean_frp",
    "max_frp",
    "spatial_extent_km",
    "nearest_osm_distance_km",
    "industrial_feature_count_1km",
    "industrial_feature_count_3km",
    "industrial_zone_count_1km",
    "factory_count_1km",
    "power_plant_count_1km",
    "works_count_1km",
    "kiln_count_1km"
]

display_cols = [c for c in display_cols if c in df.columns]

print(
    df.sort_values(
        ["active_days", "industrial_feature_count_1km"],
        ascending=False
    )[display_cols]
    .head(20)
    .round(3)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 11. HIGH OSM CONTEXT EVENTS
# ------------------------------------------------------------
print("\n[11] EVENTS WITH STRONGEST OSM CONTEXT")
print("-" * 70)

print(
    df.sort_values(
        "industrial_feature_count_1km",
        ascending=False
    )[display_cols]
    .head(20)
    .round(3)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 12. POTENTIAL ISOLATED THERMAL EVENTS
# ------------------------------------------------------------
print("\n[12] PERSISTENT EVENTS WITH LITTLE / NO OSM")
print("-" * 70)

isolated = df[
    (df["active_days"] >= 3) &
    (df["industrial_feature_count_3km"] == 0)
].copy()

print(
    f"Persistent events >=3 days with ZERO industrial OSM within 3 km:"
    f" {len(isolated):,} "
    f"({len(isolated)/len(df)*100:.2f}% of all events)"
)

if len(isolated):

    print("\nTop persistent isolated events:")

    print(
        isolated.sort_values(
            ["active_days", "max_frp"],
            ascending=False
        )[display_cols]
        .head(20)
        .round(3)
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 13. VERY STRONG INDUSTRIAL CANDIDATES
# ------------------------------------------------------------
print("\n[13] STRONG INDUSTRIAL-CONTEXT CANDIDATES")
print("-" * 70)

candidate_mask = (
    (df["active_days"] >= 3) &
    (
        (df["industrial_feature_count_1km"] > 0) |
        (df["power_plant_count_3km"] > 0) |
        (df["factory_count_3km"] > 0) |
        (df["works_count_3km"] > 0) |
        (df["kiln_count_3km"] > 0)
    )
)

industrial_candidates = df[candidate_mask]

print(
    f"Persistent + industrial-context candidates:"
    f" {len(industrial_candidates):,} "
    f"({len(industrial_candidates)/len(df)*100:.2f}%)"
)

# ------------------------------------------------------------
# 14. FINAL FEATURE CORRELATIONS
# ------------------------------------------------------------
print("\n[14] FEATURE CORRELATIONS WITH ACTIVE DAYS")
print("-" * 70)

numeric_cols = df.select_dtypes(include=np.number).columns

corr = (
    df[numeric_cols]
    .corr()["active_days"]
    .drop("active_days")
    .sort_values(key=abs, ascending=False)
)

print(corr.round(4).to_string())

# ------------------------------------------------------------
# 15. FEATURE VARIANCE
# ------------------------------------------------------------
print("\n[15] LOW-VARIANCE FEATURES")
print("-" * 70)

variance = df[numeric_cols].var().sort_values()

print(variance.head(15).round(6).to_string())

# ------------------------------------------------------------
# 16. FINAL COMPACT SUMMARY
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("FINAL COMPACT SUMMARY")
print("=" * 70)

print(f"Total events                         : {len(df):,}")
print(f"Events active >=2 days               : {(df.active_days >= 2).sum():,}")
print(f"Events active >=3 days               : {(df.active_days >= 3).sum():,}")
print(f"Events active >=5 days               : {(df.active_days >= 5).sum():,}")
print(f"Events active >=7 days               : {(df.active_days >= 7).sum():,}")

print(
    f"Events with OSM within 1 km          : "
    f"{(df.industrial_feature_count_1km > 0).sum():,}"
)

print(
    f"Events with OSM within 3 km          : "
    f"{(df.industrial_feature_count_3km > 0).sum():,}"
)

print(
    f"Persistent >=3d + OSM context        : "
    f"{len(industrial_candidates):,}"
)

print(
    f"Persistent >=3d + NO OSM within 3km  : "
    f"{len(isolated):,}"
)

print(
    f"Median nearest OSM distance (km)     : "
    f"{df.nearest_osm_distance_km.median():.3f}"
)

print(
    f"Maximum event spatial extent (km)    : "
    f"{df.spatial_extent_km.max():.3f}"
)

print("\n" + "=" * 70)
print("END OF DIAGNOSTIC")
print("=" * 70)

V8 COMPLETE DIAGNOSTIC SUMMARY

[1] DATASET OVERVIEW
----------------------------------------------------------------------
Rows / events              : 3,337
Columns                    : 48
Missing values (total)     : 0
Duplicate event IDs        : 0

Date range:
  Start                     : 2024-01-01
  End                       : 2024-01-31

[2] TEMPORAL EVENT CHARACTERISTICS
----------------------------------------------------------------------

active_days:
count   3337.0000
mean       1.9870
std        3.4310
min        1.0000
25%        1.0000
50%        1.0000
75%        1.0000
max       31.0000

duration_days:
count   3337.0000
mean       2.3130
std        4.3420
min        1.0000
25%        1.0000
50%        1.0000
75%        1.0000
max       31.0000

detection_count:
count   3337.0000
mean       4.0780
std       17.6590
min        1.0000
25%        1.0000
50%        1.0000
75%        2.0000
max      714.0000

activity_frequency:
count   3337.0000
mean       0.9650
std     